# Part 3: AdaBoost (5.0 points)

Recall the AdaBoost algorithm:

**Given**: $(\mathbf{x}_1, y_1), \dots, (\mathbf{x}_m, y_m)$ where $\mathbf{x}_i \in \mathbb{R}^d$ and $y_i \in \{-1, +1\}$

**Initialize**: $D_1(i) = 1/m$ for $i = 1, \dots, m$

**For** $t = 1, \dots, T$:
1. Train weak learner $h_t$ minimizing the weighted training error:
$$\varepsilon_t = \sum_{i=1}^m D_t(i) \mathbf{1}\big(h_t(\mathbf{x}_i) \neq y_i\big)$$
2. Choose step-size:
$$\alpha_t = \dfrac{1}{2}\log\left(\dfrac{1-\varepsilon_t}{\varepsilon_t}\right)$$
3. Update weights:
$$D_{t+1}(i) = \dfrac{D_t(i)\exp(-\alpha_t y_i h_t(\mathbf{x}_i))}{Z_t}$$
where $Z_t$ is a normalization factor.

**Output**: $H(\mathbf{x}) = \text{sign}\left(\sum_{t=1}^T \alpha_t h_t(\mathbf{x})\right)$

---
The figure in the exam PDF shows three iterations of AdaBoost using a depth-1 decision tree (decision stump) on a given dataset.

**(a)** For each iteration in the figure, find the weighted training error εₜ and importance αₜ of hₜ. For t = 2 and t = 3, find the weight normalization Zₜ and record the updated weight for each point. **(1 point)**


---
## (a) Manual weight computation (1.0 point)

For each iteration in the figure, find the weighted training error $\varepsilon_t$ and importance $\alpha_t$ of $h_t$. For $t = 2$ and $t = 3$, find the weight normalization $Z_t$ and the updated weight for each point.

In [ ]:
import numpy as np

# Iteration 1
w_1 = np.ones(11)
Z_1 = np.sum(w_1)
w_1 = w_1 / Z_1

y_true = np.array([+1, +1, -1, -1, -1, -1, +1, +1, +1, -1, -1])
y_pred_1 = np.array([+1, +1, -1, -1, -1, -1, -1, -1, -1, -1, -1])
e_1 = np.dot(w_1, y_pred_1 != y_true)
alpha_1 = 1/2 * np.log((1 - e_1)/e_1)

print("Iteration 1")
print("e_1:", e_1)
print("alpha_1:", alpha_1)
print("Z_1:", Z_1)

# Iteration 2
w_2 = np.multiply(w_1, np.exp(-alpha_1 * np.multiply(y_true, y_pred_1)))
Z_2 = np.sum(w_2)
w_2 = w_2 / Z_2

y_pred_2 = np.array([+1, -1, -1, +1, -1, -1, +1, +1, +1, +1, -1])
e_2 = np.dot(w_2, y_pred_2 != y_true)
alpha_2 = 1/2 * np.log((1 - e_2)/e_2)

print("\nIteration 2")
print("e_2:", e_2)
print("alpha_2:", alpha_2)
print("Z_2:", Z_2)
print("w_2:", w_2)

# Iteration 3
w_3 = np.multiply(w_2, np.exp(-alpha_2 * np.multiply(y_true, y_pred_2)))
Z_3 = np.sum(w_3)
w_3 = w_3 / Z_3

y_pred_3 = np.array([+1, +1, +1, +1, +1, +1, +1, +1, +1, -1, -1])
e_3 = np.dot(w_3, y_pred_3 != y_true)
alpha_3 = 1/2 * np.log((1 - e_3)/e_3)

print("\nIteration 3")
print("e_3:", e_3)
print("alpha_3:", alpha_3)
print("Z_3:", Z_3)
print("w_3:", w_3)

**(b)** Load the `datasets/dataset_ex3.csv` file into a dataframe and make a scatter plot of the data points using the same markers and colors as in the figure (hint: `marker='+'` for positive class, `marker='_'` for negative class in matplotlib). **(1 point)**


---
## (b) Scatter plot of the dataset (1.0 point)

Load `../datasets/dataset_ex3.csv` and make a scatter plot using `marker='+'` for positive labels (blue) and `marker='_'` for negative labels (red).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv('../datasets/dataset_ex3.csv')
X = df[['x1', 'x2']].values
y = df['label']
y = np.where(y == 'plus', 1, -1)  # Convert labels to {-1, +1}

fig, ax = plt.subplots(figsize=(4, 4))
idx_plus = np.where(y == 1)[0]
ax.scatter(X[idx_plus, 0], X[idx_plus, 1], c='b', marker='+', s=100)
idx_minus = np.where(y == -1)[0]
ax.scatter(X[idx_minus, 0], X[idx_minus, 1], c='r', marker='_', s=100)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('Dataset (blue=+1, red=-1)')
plt.show()

**(c)** With the help of `DecisionTreeClassifier` from sklearn, fit a decision stump h₁ on the dataset corresponding to the first iteration of AdaBoost. Remember to set a uniform weight w₁ over all data points and ensure labels yᵢ ∈ {−1, +1}. Show that your predictions match those from the first iteration in the figure. **(1 point)**


---
## (c) Fit decision stump h1 with uniform weights (1.0 point)

Using `DecisionTreeClassifier` from `sklearn`, fit a decision stump $h_1$ with uniform weights over all data points. Show that your predictions match those from the first iteration in the exam figure.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

w_1 = np.ones(len(y))
Z_1 = np.sum(w_1)
w_1 = w_1 / Z_1

clf_1 = DecisionTreeClassifier(max_depth=1)
clf_1.fit(X, y, sample_weight=w_1)
y_pred_1 = clf_1.predict(X)
print('h1 predictions:', y_pred_1)

**(d)** Calculate the new weights w₂ of the data points based on predictions from iteration t = 1. Use them to fit a new decision stump h₂ on the weighted dataset (hint: use the optional argument `sample_weight` in sklearn's `fit` method). Show that your predictions match those from the second iteration in the figure. **(0.5 points)**


---
## (d) Compute weights w2 and fit decision stump h2 (0.5 point)

Calculate the new weights $w_2$ based on the predictions from iteration $t = 1$. Fit a new decision stump $h_2$ and show that your predictions match those from the second iteration.

In [ ]:
e_1 = np.dot(w_1, y_pred_1 != y)
alpha_1 = 1/2 * np.log((1 - e_1)/e_1)

w_2 = np.multiply(w_1, np.exp(-alpha_1 * np.multiply(y, y_pred_1)))
Z_2 = np.sum(w_2)
w_2 = w_2 / Z_2

clf_2 = DecisionTreeClassifier(max_depth=1)
clf_2.fit(X, y, sample_weight=w_2)
y_pred_2 = clf_2.predict(X)
print('h2 predictions:', y_pred_2)
print('alpha_1:', alpha_1)

**(e)** Calculate the new weights w₃ of the data points based on predictions from iteration t = 2. Use them to fit a new decision stump h₃ on the weighted dataset. Show that your predictions match those from the third iteration in the figure. **(0.5 points)**


---
## (e) Compute weights w3 and fit decision stump h3 (0.5 point)

Calculate the new weights $w_3$ based on the predictions from iteration $t = 2$. Fit a new decision stump $h_3$ and show that your predictions match those from the third iteration.

In [ ]:
e_2 = np.dot(w_2, y_pred_2 != y)
alpha_2 = 1/2 * np.log((1 - e_2)/e_2)

w_3 = np.multiply(w_2, np.exp(-alpha_2 * np.multiply(y, y_pred_2)))
Z_3 = np.sum(w_3)
w_3 = w_3 / Z_3

clf_3 = DecisionTreeClassifier(max_depth=1)
clf_3.fit(X, y, sample_weight=w_3)
y_pred_3 = clf_3.predict(X)
print('h3 predictions:', y_pred_3)
print('alpha_2:', alpha_2)

e_3 = np.dot(w_3, y_pred_3 != y)
alpha_3 = 1/2 * np.log((1 - e_3)/e_3)
print('alpha_3:', alpha_3)

**(f)** Calculate the training error of the final ensemble classifier, i.e. the classifier that for each datapoint x outputs H(x) = sign(α₁h₁(x) + α₂h₂(x) + α₃h₃(x)). **(1 point)**

---


---
## (f) Training error of the final ensemble (1.0 point)

Calculate the training error of the final ensemble classifier:
$$H(\mathbf{x}) = \text{sign}\big(\alpha_1 h_1(\mathbf{x}) + \alpha_2 h_2(\mathbf{x}) + \alpha_3 h_3(\mathbf{x})\big)$$

In [ ]:
y_1 = clf_1.predict(X)
y_2 = clf_2.predict(X)
y_3 = clf_3.predict(X)
y_final = np.sign(alpha_1 * y_1 + alpha_2 * y_2 + alpha_3 * y_3)
training_error = np.mean(y_final != y)
print('Training error of the ensemble:', training_error)